# Ollama en SageMaker (m5.xlarge)
1. Crear la *Lifecycle configuration* **on-start** con el contenido de `sagemaker/on-start.sh` y asociarla al notebook.
2. Crear el notebook instance `ml.m5.xlarge` (volumen ≥ 30 GB) y abrir JupyterLab.
3. Ejecutar estas celdas: verifican Docker, `data-root`, el contenedor, el modelo y una respuesta real.

Todo lo persistente vive en `/home/ec2-user/SageMaker`.

In [ ]:
!docker info 2>/dev/null | grep "Docker Root Dir"   # debe ser /home/ec2-user/SageMaker/docker-data
!docker ps --filter name=ollama

Si es la primera vez y la lifecycle aún no corrió, se puede lanzar a mano:

In [ ]:
# !sudo bash /home/ec2-user/SageMaker/LabDino/sagemaker/on-start.sh

In [ ]:
MODEL = "gemma4:e2b"
!docker exec ollama ollama pull {MODEL}
!docker exec ollama ollama list

In [ ]:
import requests, json
BASE = "http://localhost:11434"
print("Versión de Ollama:", requests.get(f"{BASE}/api/version").json())
info = requests.post(f"{BASE}/api/show", json={"model": MODEL}).json()
print(json.dumps(info.get("details"), indent=2))

### Prueba: el modelo recibe una instrucción y responde

In [ ]:
r = requests.post(f"{BASE}/api/generate", json={"model": MODEL, "prompt": "Di hola en una frase como un dinosaurio.",
                                                "stream": False, "think": False}, timeout=600)
print(r.json()["response"])

### Registrar modelo e identidad con el código del repo
Clonar el repo dentro de `/home/ec2-user/SageMaker` (persistente) e instalar el paquete.

In [ ]:
%cd /home/ec2-user/SageMaker
!test -d LabDino || git clone https://github.com/cofferr/LabDino
%cd LabDino
!pip install -q -e .
import os
os.environ["OLLAMA_BASE_URL"] = BASE
os.environ["OLLAMA_MODEL"] = MODEL
!python ollama/write_model_info.py
!python ollama/generate_identity.py

### Exponer Ollama a la app web (ngrok)
Guardar el token **fuera del repo**: `echo TU_TOKEN > /home/ec2-user/SageMaker/.ngrok_token` (la lifecycle lo arranca en cada inicio).
La URL pública se lee aquí y va en `OLLAMA_BASE_URL` del backend, con `OLLAMA_NGROK_SKIP_WARNING=true`.

In [ ]:
import requests
try:
    tunnels = requests.get("http://localhost:4040/api/tunnels").json()["tunnels"]
    print([t["public_url"] for t in tunnels])
except Exception as e:
    print("ngrok no está activo:", e)